# Single line-of-sight diagnostics — three test datasets

Walk one ray at a time and verify the ray tracing: the per-segment profile (`plot_los_profile`) and the r/θ/φ voxel staircase (`plot_los_voxel_proof`). A **PASS** verdict means every kept sample lies inside its assigned cell *and* the geometry residuals are ~0 (closest approach = impact parameter, r(s) matches √(b² + (s−s₀)²), spherical↔Cartesian round-trip exact).

Each test case is a YAML config in `examples/configs/` — the notebook loads the *same* file `rrt-los` does, probes included.

| `CASE` | data | config | notes |
|---|---|---|---|
| `case1` | `rho_bin.h5` + `t_bin.h5` | [`case1_los.yaml`](../configs/case1_los.yaml) | MAS/PSI full shell, real T field, sub-Earth view |
| `case2` | `mhd_data_0220.h5` | [`case2_los.yaml`](../configs/case2_los.yaml) | ARMS 90° wedge, density-only → isothermal 1 MK |
| `case3` | `rhoC_02.h5` + `t02.h5` | [`case3_los.yaml`](../configs/case3_los.yaml) | MAS/PSI full shell, different snapshot |

**Choosing probes.** `(X, Y)` are impact points on the sky in R☉. `b = √(X²+Y²)` must be smaller than the cube's outer radius, and for a partial domain the ray has to actually cross it — at the wedge's side view only `X ≳ 0.15` does, so a probe like `(0.5, 0)` that works for the MAS shells has zero kept samples there. Probes with `b < 1` are on-disk and get clipped at the photosphere; `b > 1` sees the full chord.

In [ ]:
import os
from pathlib import Path

import numpy as np, matplotlib.pyplot as plt
from rrt.config import load_config, load_cube_from_config
from rrt.observer import observer_from_view
from rrt.response import load_response
from rrt.los import sample_along_los, plot_los_profile, plot_los_voxel_proof

# configs use repo-relative paths (data/…), so work from the repo root
REPO = Path.cwd().resolve()
while not (REPO / 'examples' / 'configs').exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)
print('working from:', REPO)

## 1. Pick the test case

Change `CASE` and re-run everything below. The probes come from the config's `los:` block — reassign `PROBES` by hand if you want to explore other pixels.

In [ ]:
CASE = 'case1'          # 'case1' | 'case2' | 'case3'

CONFIGS = {
    'case1':    'examples/configs/case1_los.yaml',
    'case2': 'examples/configs/case2_los.yaml',
    'case3':    'examples/configs/case3_los.yaml',
}

cfg = load_config(CONFIGS[CASE])
cube = load_cube_from_config(cfg)
print(cube.summary())
r, th, ph, ne, T = cube.r, cube.theta, cube.phi, cube.ne, cube.T

T_resp, R_resp = load_response(cfg.response, cfg.wavelength)
n_obs, e_los, x_img, y_img = observer_from_view(cfg.views[0])

PROBES = [tuple(p) for p in cfg.los.get('probes', [[1.5, 0.3]])]
print(f"{CASE}: view '{cfg.views[0]['label']}', AIA {cfg.wavelength} Å, probes {PROBES}")

## 2. Walk each ray — profile + voxel proof

`sample_along_los` splits the ray on the *same* boundary surfaces the production integrators use and assigns cells with the identical `searchsorted(edges, x, 'right') - 1` rule, so the shaded bands in the staircase are the voxels actually read. Two figures per probe; both are also written to the config's `output_dir` with an `nb_` prefix.

In [ ]:
OUT_DIR = Path(cfg.output_dir); OUT_DIR.mkdir(parents=True, exist_ok=True)
samples = {}

for X, Y in PROBES:
    s = sample_along_los(X, Y, x_img, y_img, e_los, r, th, ph, ne, T,
                         T_resp=T_resp, R_resp=R_resp)
    if s['kept'].sum() == 0:
        print(f'({X:+.2f},{Y:+.2f})  SKIPPED — this LOS never enters the domain')
        continue
    samples[(X, Y)] = s
    tag = f'{X:+.2f}_{Y:+.2f}'.replace('.', 'p').replace('+', '')

    fig, _ = plot_los_profile(s)
    fig.savefig(OUT_DIR / f'nb_los_profile_{tag}.png', dpi=150, bbox_inches='tight')

    vp = plot_los_voxel_proof(X, Y, x_img, y_img, e_los, r, th, ph, ne, T,
                              samples=s, show=False,
                              savepath=str(OUT_DIR / f'nb_los_voxel_{tag}.png'))
    s['voxel_proof'] = vp
    print(f"({X:+.2f},{Y:+.2f})  kept={int(s['kept'].sum())}/{s['kept'].size}  "
          f"Σ={s.get('I_total_clipped', s['I_total']):.3e} DN/s/pix  "
          f"inside={vp['frac_inside_pct']:.1f}%  verdict={'PASS' if vp['pass'] else 'CHECK'}")

## 3. Cross-check against the production integrator

The real test of the walker: its per-segment sum must reproduce `rrt.euv.run_siddon` for the same pixel. Rendering a 1×1 image at each probe should agree to round-off — if it doesn't, the diagnostic and the integrator disagree about geometry or cell assignment.

In [ ]:
from rrt.euv import run_siddon

print(f'{"probe":>16s}  {"LOS sum":>12s}  {"integrator":>12s}  {"rel.diff":>9s}')
for (X, Y), s in samples.items():
    pix = run_siddon(np.array([[X]]), np.array([[Y]]), x_img, y_img, e_los,
                     r, th, ph, ne, T, T_resp, R_resp,
                     Npix=1, Rmax=1.0, plot=False, save=False)[0, 0]
    tot = s.get('I_total_clipped', s['I_total'])
    rel = abs(pix - tot) / tot if tot else np.nan
    print(f'({X:+.2f},{Y:+.2f})   {tot:12.4e}  {pix:12.4e}  {rel:9.1e}')